In [2]:
#Vamos a intentar balancear las clases de las muestras por paises, para ello empezaremos a reducir muestras de paises con exceso de muestras. Luego pasaremos el dataframe resultante a un analisis descriptivo.

import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/UE128k.csv')

In [4]:
df['cntry'].value_counts()

cntry
Alemania        31012
Países Bajos    17017
Bélgica         16906
España          15721
Hungría         12565
Portugal        11678
Eslovaquia       9181
Italia           8617
Croacia          5866
Name: count, dtype: int64

In [5]:
#Objetivo del balanceo de muestras es mantener las proporciones interclase respecto a nuestra variable target lrscale. 
#Aproximaremos el numero de muestras por pais a 10k, lo conseguiremos con el siguiente codigo.
#Primero aplicamos un submuestreo para eliminar clases mayoritarias agrupadas por paises.
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit



target_size = 10000  # Tamaño deseado para cada país

df1 = pd.DataFrame()  # DataFrame para los datos balanceados

# Imprimir información sobre las columnas
print("Información sobre las columnas:")
print(f"Columna objetivo: {df['lrscale'].name}, Tipo: {df['lrscale'].dtype}, Valores únicos: {df['lrscale'].unique()}")
print(f"Columna de país: {df['cntry'].name}, Tipo: {df['cntry'].dtype}, Valores únicos: {df['cntry'].unique()}")

for country in df['cntry'].unique():
    print(f"\nProcesando país: {country}")
    df_country = df[df['cntry'] == country]
    print(f"Tamaño de df_country: {len(df_country)}")

    if len(df_country) > target_size:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=target_size, random_state=42)
        train_index, _ = next(sss.split(df_country, df_country['lrscale']))
        df_country_sampled = df_country.iloc[train_index]
    else:
        df_country_sampled = df_country

    df1 = pd.concat([df1, df_country_sampled])

print("\nConteo de países en el DataFrame balanceado:")
print(df1['cntry'].value_counts())

Información sobre las columnas:
Columna objetivo: lrscale, Tipo: float64, Valores únicos: [ 7.  5.  2.  3.  1.  0.  6.  4. 10.  9.  8.]
Columna de país: cntry, Tipo: object, Valores únicos: ['Bélgica' 'Alemania' 'Países Bajos' 'España' 'Italia' 'Portugal'
 'Croacia' 'Hungría' 'Eslovaquia']

Procesando país: Bélgica
Tamaño de df_country: 16906

Procesando país: Alemania
Tamaño de df_country: 31012

Procesando país: Países Bajos
Tamaño de df_country: 17017

Procesando país: España
Tamaño de df_country: 15721

Procesando país: Italia
Tamaño de df_country: 8617

Procesando país: Portugal
Tamaño de df_country: 11678

Procesando país: Croacia
Tamaño de df_country: 5866

Procesando país: Hungría
Tamaño de df_country: 12565

Procesando país: Eslovaquia
Tamaño de df_country: 9181

Conteo de países en el DataFrame balanceado:
cntry
Bélgica         10000
Alemania        10000
Países Bajos    10000
España          10000
Portugal        10000
Hungría         10000
Eslovaquia       9181
Italia      

In [6]:
print(df1['cntgrp_fc'].value_counts())

cntgrp_fc
1    34483
0    30000
2    19181
Name: count, dtype: int64


In [ ]:
#Siguiente paso es calibrar el submuestreo y la estratificacion para que nos de un dataset balanceado de muestras por pais.
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.over_sampling import SMOTE

def balance_dataframe(df, target_size=10000, reduction_group0=9000, reduction_group1=12000, increase_group2=5000, random_state=42):
    df_balanced = pd.DataFrame()  # DataFrame para los datos balanceados
    group0 = ['Alemania', 'Bélgica', 'Países Bajos']
    group1 = ['España', 'Portugal', 'Italia', 'Croacia']
    group2 = ['Eslovaquia', 'Hungría']

    for country in df['cntry'].unique():
        df_country = df[df['cntry'] == country]
        n_samples = min(target_size, len(df_country))

        if country in group0:
            n_samples = max(0, n_samples - reduction_group0 // len(group0))
        elif country in group1:
            n_samples = max(0, n_samples - reduction_group1 // len(group1))
        elif country in group2:
            n_samples = min(target_size + increase_group2 // len(group2), len(df_country))

        print(f"\nProcesando país: {country}")
        print(f"Tamaño original: {len(df_country)}")
        print(f"Tamaño de submuestreo/sobremuestreo: {n_samples}")

        if n_samples < len(df_country):  
            sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=random_state)
            for train_index, _ in sss.split(df_country, df_country['lrscale']):
                print(f"Índices de entrenamiento: {train_index}")
                df_country_sampled = df_country.iloc[train_index]
        elif country in group2 and n_samples > len(df_country):  
            smote = SMOTE(random_state=random_state)
            X_resampled, y_resampled = smote.fit_resample(df_country.drop('lrscale', axis=1), df_country['lrscale'])
            print(f"Dimensiones de datos generados por SMOTE: {X_resampled.shape}")
            df_country_sampled = pd.concat([pd.DataFrame(X_resampled, columns=df_country.drop('lrscale', axis=1).columns), pd.Series(y_resampled, name='lrscale')], axis=1)
        else:
            df_country_sampled = df_country

        print(f"Tamaño de df_country_sampled: {len(df_country_sampled)}")
        df_balanced = pd.concat([df_balanced, df_country_sampled])

    return df_balanced

# Ejemplo de uso
df_balanced = balance_dataframe(df1)
print(df_balanced['cntry'].value_counts())



Procesando país: Bélgica
Tamaño original: 10000
Tamaño de submuestreo/sobremuestreo: 7000
Índices de entrenamiento: [7776 3186 4924 ... 2927 9434 9369]
Tamaño de df_country_sampled: 7000

Procesando país: Alemania
Tamaño original: 10000
Tamaño de submuestreo/sobremuestreo: 7000
Índices de entrenamiento: [ 833 2414 2089 ... 1665 7044 9850]
Tamaño de df_country_sampled: 7000

Procesando país: Países Bajos
Tamaño original: 10000
Tamaño de submuestreo/sobremuestreo: 7000
Índices de entrenamiento: [6198 3638 5749 ... 1149 8396 7197]
Tamaño de df_country_sampled: 7000

Procesando país: España
Tamaño original: 10000
Tamaño de submuestreo/sobremuestreo: 7000
Índices de entrenamiento: [9565 7789 2100 ... 1371 8467 2016]
Tamaño de df_country_sampled: 7000

Procesando país: Italia
Tamaño original: 8617
Tamaño de submuestreo/sobremuestreo: 5617
Índices de entrenamiento: [ 628 2490 4567 ... 3406 4062 4731]
Tamaño de df_country_sampled: 5617

Procesando país: Portugal
Tamaño original: 10000
Tamaño 

In [9]:

df_balanced.reset_index(drop=True, inplace=True)

In [10]:
df_balanced.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/dfeda.csv',index=False)